In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='project-8e6c1255-803c-4395-88f')

def mask_s2_clouds(image):
  """Masks clouds in a Sentinel-2 image using the QA band.

  Args:
      image (ee.Image): A Sentinel-2 image.

  Returns:
      ee.Image: A cloud-masked Sentinel-2 image.
  """
  qa = image.select('QA60')

  # Bits 10 and 11 are clouds and cirrus, respectively.
  cloud_bit_mask = 1 << 10
  cirrus_bit_mask = 1 << 11

  # Both flags should be set to zero, indicating clear conditions.
  mask = (
      qa.bitwiseAnd(cloud_bit_mask)
      .eq(0)
      .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
  )

  return image.updateMask(mask).divide(10000)

# Region of interest
roi = ee.Geometry.Rectangle([83.25, 17.69, 83.31, 17.73])



dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2020-01-01', '2020-01-30')
    # Pre-filter to get less cloudy granules.
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .filterBounds(roi)
    .map(mask_s2_clouds)
)

def add_region_stats(image):
    stats = image.select(['B2', 'B3', 'B4', 'B8']).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,
        maxPixels=1e9
    )

    image_date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')

    return image.set(stats).set({
        'date': image_date,
        'image_id': image.id()
    })

with_stats = dataset.map(add_region_stats)


visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
}

m = geemap.Map()
m.set_center(83.277, 17.7009, 12)
m.add_layer(dataset.mean(), visualization, 'RGB')
m

Map(center=[17.7009, 83.277], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…

In [8]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='project-8e6c1255-803c-4395-88f')

# Optional region of interest
roi = ee.Geometry.Rectangle([83.25, 17.69, 83.31, 17.73])

# 1) RAW collection: keep original metadata
raw_dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2020-01-01', '2020-01-30')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

# 2) Processed collection: explicitly preserve properties
def mask_s2_clouds(image):
    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    processed = image.updateMask(mask).divide(10000)

    # Copy all properties from the source image to the processed image
    return processed.copyProperties(image, image.propertyNames())

processed_dataset = raw_dataset.map(mask_s2_clouds)

In [9]:
raw_img = raw_dataset.first()

print("RAW property names:")
print(raw_img.propertyNames().getInfo())

print("RAW cloudiness:")
print(raw_img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo())

print("RAW properties dictionary:")
print(raw_img.toDictionary().getInfo())

RAW property names:
['system:version', 'system:id', 'DATATAKE_IDENTIFIER', 'AOT_RETRIEVAL_ACCURACY', 'SPACECRAFT_NAME', 'SATURATED_DEFECTIVE_PIXEL_PERCENTAGE', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B8A', 'CLOUD_SHADOW_PERCENTAGE', 'MEAN_SOLAR_AZIMUTH_ANGLE', 'system:footprint', 'VEGETATION_PERCENTAGE', 'SOLAR_IRRADIANCE_B12', 'SOLAR_IRRADIANCE_B10', 'SENSOR_QUALITY', 'SOLAR_IRRADIANCE_B11', 'GENERATION_TIME', 'SOLAR_IRRADIANCE_B8A', 'FORMAT_CORRECTNESS', 'CLOUD_COVERAGE_ASSESSMENT', 'THIN_CIRRUS_PERCENTAGE', 'system:time_end', 'WATER_VAPOUR_RETRIEVAL_ACCURACY', 'system:time_start', 'DATASTRIP_ID', 'PROCESSING_BASELINE', 'SENSING_ORBIT_NUMBER', 'NODATA_PIXEL_PERCENTAGE', 'SENSING_ORBIT_DIRECTION', 'GENERAL_QUALITY', 'GRANULE_ID', 'REFLECTANCE_CONVERSION_CORRECTION', 'MEDIUM_PROBA_CLOUDS_PERCENTAGE', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B8', 'DATATAKE_TYPE', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B9', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B6', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B7', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B4', 'MEAN_

In [10]:
img = processed_dataset.first()

print("Processed property names:")
print(img.propertyNames().getInfo())

print("Processed cloudiness:")
print(img.get('CLOUDY_PIXEL_PERCENTAGE').getInfo())

print("Processed properties dict:")
print(img.toDictionary().getInfo())

Processed property names:
['system:id', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B8A', 'CLOUD_SHADOW_PERCENTAGE', 'MEAN_SOLAR_AZIMUTH_ANGLE', 'VEGETATION_PERCENTAGE', 'SOLAR_IRRADIANCE_B12', 'system:version', 'SOLAR_IRRADIANCE_B10', 'SENSOR_QUALITY', 'SOLAR_IRRADIANCE_B11', 'GENERATION_TIME', 'SOLAR_IRRADIANCE_B8A', 'FORMAT_CORRECTNESS', 'CLOUD_COVERAGE_ASSESSMENT', 'THIN_CIRRUS_PERCENTAGE', 'system:time_end', 'WATER_VAPOUR_RETRIEVAL_ACCURACY', 'system:time_start', 'DATASTRIP_ID', 'PROCESSING_BASELINE', 'SENSING_ORBIT_NUMBER', 'NODATA_PIXEL_PERCENTAGE', 'SENSING_ORBIT_DIRECTION', 'GENERAL_QUALITY', 'GRANULE_ID', 'REFLECTANCE_CONVERSION_CORRECTION', 'MEDIUM_PROBA_CLOUDS_PERCENTAGE', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B8', 'DATATAKE_TYPE', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B9', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B6', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B7', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B4', 'MEAN_INCIDENCE_ZENITH_ANGLE_B1', 'NOT_VEGETATED_PERCENTAGE', 'MEAN_INCIDENCE_AZIMUTH_ANGLE_B5', 'RADIOMETRIC_QUALITY', 'MEAN_IN

In [11]:
sentinel_props = [
    'AOT_RETRIEVAL_ACCURACY',
    'CLOUDY_PIXEL_PERCENTAGE',
    'CLOUD_COVERAGE_ASSESSMENT',
    'CLOUDY_SHADOW_PERCENTAGE',
    'DARK_FEATURES_PERCENTAGE',
    'DATASTRIP_ID',
    'DATATAKE_IDENTIFIER',
    'DATATAKE_TYPE',
    'DEGRADED_MSI_DATA_PERCENTAGE',
    'FORMAT_CORRECTNESS',
    'GENERAL_QUALITY',
    'GENERATION_TIME',
    'GEOMETRIC_QUALITY',
    'GRANULE_ID',
    'HIGH_PROBA_CLOUDS_PERCENTAGE',
    'MEAN_SOLAR_AZIMUTH_ANGLE',
    'MEAN_SOLAR_ZENITH_ANGLE',
    'MEDIUM_PROBA_CLOUDS_PERCENTAGE',
    'MGRS_TILE',
    'NODATA_PIXEL_PERCENTAGE',
    'NOT_VEGETATED_PERCENTAGE',
    'PROCESSING_BASELINE',
    'PRODUCT_ID',
    'RADIATIVE_TRANSFER_ACCURACY',
    'RADIOMETRIC_QUALITY',
    'REFLECTANCE_CONVERSION_CORRECTION',
    'SATURATED_DEFECTIVE_PIXEL_PERCENTAGE',
    'SENSING_ORBIT_DIRECTION',
    'SENSING_ORBIT_NUMBER',
    'SENSOR_QUALITY',
    'SNOW_ICE_PERCENTAGE',
    'SPACECRAFT_NAME',
    'THIN_CIRRUS_PERCENTAGE',
    'UNCLASSIFIED_PERCENTAGE',
    'VEGETATION_PERCENTAGE',
    'WATER_PERCENTAGE',
    'WATER_VAPOUR_RETRIEVAL_ACCURACY',
    'system:index',
    'system:time_start'
]

In [14]:
def image_to_feature(image):
    return ee.Feature(None, image.toDictionary(sentinel_props))

property_table = ee.FeatureCollection(raw_dataset.map(image_to_feature))


In [15]:
def add_region_stats(image):
    stats = image.select(['B2', 'B3', 'B4', 'B8', 'B11']).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,
        maxPixels=1e9
    )

    return image.set({
        'roi_B2_mean': stats.get('B2'),
        'roi_B3_mean': stats.get('B3'),
        'roi_B4_mean': stats.get('B4'),
        'roi_B8_mean': stats.get('B8'),
        'roi_B11_mean': stats.get('B11')
    })

raw_with_stats = raw_dataset.map(add_region_stats)

In [17]:
full_props = sentinel_props + [
    'roi_B2_mean', 'roi_B3_mean', 'roi_B4_mean', 'roi_B8_mean', 'roi_B11_mean'
]

def image_to_feature_with_stats(image):
    return ee.Feature(None, image.toDictionary(full_props))

full_table = ee.FeatureCollection(raw_with_stats.map(image_to_feature_with_stats))

In [19]:
import pandas as pd

df = ee.data.computeFeatures({
    'expression': full_table,
    'fileFormat': 'PANDAS_DATAFRAME'
})

df.to_csv('s2_metadata_and_roi_stats.csv', index=False)
df.head()

,geo,AOT_RETRIEVAL_ACCURACY,CLOUDY_PIXEL_PERCENTAGE,CLOUD_COVERAGE_ASSESSMENT,DARK_FEATURES_PERCENTAGE,DATASTRIP_ID,DATATAKE_IDENTIFIER,DATATAKE_TYPE,DEGRADED_MSI_DATA_PERCENTAGE,FORMAT_CORRECTNESS,...,THIN_CIRRUS_PERCENTAGE,UNCLASSIFIED_PERCENTAGE,VEGETATION_PERCENTAGE,WATER_PERCENTAGE,WATER_VAPOUR_RETRIEVAL_ACCURACY,roi_B11_mean,roi_B2_mean,roi_B3_mean,roi_B4_mean,roi_B8_mean
0,None,0,1.726278,1.726278,0.435449,S2B_OPER_MSI_L2A_DS_EPAE_20200108T081052_S2020...,GS2B_20200108T045159_014831_N02.13,INS-NOBS,0.0000,PASSED,...,0.000484,5.323622,12.427571,72.271061,0,1618.671298,765.057436,848.969026,867.908235,1477.912826
1,None,0,0.040560,0.040560,0.023167,S2A_OPER_MSI_L2A_DS_S2RP_20230425T185402_S2020...,GS2A_20200113T045131_023811_N05.00,INS-NOBS,0.0096,PASSED,...,0.000298,0.033530,13.914184,74.048358,0,1640.982951,914.487671,993.688347,1012.764027,1637.732454
2,None,0,5.175906,5.175906,0.670034,S2B_OPER_MSI_L2A_DS_EPAE_20200118T084613_S2020...,GS2B_20200118T045129_014974_N02.13,INS-NOBS,0.0000,PASSED,...,0.004593,2.437210,13.082631,69.518679,0,1611.616664,712.668922,795.498911,826.121298,1411.320809
3,None,0,3.433870,3.433870,0.320488,S2A_OPER_MSI_L2A_DS_EPAE_20200123T084215_S2020...,GS2A_20200123T045101_023954_N02.13,INS-NOBS,0.0000,PASSED,...,0.000325,9.193598,6.758954,72.316301,0,1732.327626,1128.641347,1197.192769,1182.609284,1632.339578


In [3]:
# Load Landsat 8 data, filter by date, month, and bounds.
collection = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
    # Three years of data
    .filterDate('2015-01-01', '2018-01-01')
    # Only Nov-Feb observations
    .filter(ee.Filter.calendarRange(11, 2, 'month'))
    # Intersecting ROI
    .filterBounds(ee.Geometry.Point(25.8544, -18.08874))
)

# Also filter the collection by the CLOUD_COVER property.
filtered = collection.filter(ee.Filter.eq('CLOUD_COVER', 0))

# Create two composites to check the effect of filtering by CLOUD_COVER.
bad_composite = collection.mean()
good_composite = filtered.mean()

# Display the composites.
m = geemap.Map()
m.set_center(25.8544, -18.08874, 13)
m.add_layer(
    bad_composite,
    {'bands': ['B3', 'B2', 'B1'], 'min': 0.05, 'max': 0.35, 'gamma': 1.1},
    'Bad composite',
)
m.add_layer(
    good_composite,
    {'bands': ['B3', 'B2', 'B1'], 'min': 0.05, 'max': 0.35, 'gamma': 1.1},
    'Good composite',
)
m

Map(center=[-18.08874, 25.8544], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

In [ ]:
import ee
import geemap
import pandas as pd

# authenentication
ee.Authenticate()
ee.Initialize(project='your-real-project-id')   # replace this

# coordinates + radius
lon = 8.4064
lat = 46.5763

radius = 3000

# Time range
start_date = '2020-06-01'
end_date = '2020-09-30'

# Cloud filter
max_cloud = 30

# Bands for ROI statistics
bands_for_stats = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']

# Output names
drive_folder = 'GEE_exports'
drive_filename = 'switzerland_glacier_s2_metadata_stats'
local_csv = 'switzerland_glacier_s2_metadata_stats.csv'

# region of interest
def roi_from_point(lon, lat, radius=3000):
    point = ee.Geometry.Point([lon, lat])
    roi = point.buffer(radius).bounds()
    return roi

roi = roi_from_point(lon, lat, radius)

# country specific data
countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
switzerland = countries.filter(ee.Filter.eq('country_na', 'Switzerland')).geometry()

roi = roi.intersection(switzerland, ee.ErrorMargin(1))

#cloud mask function
def mask_s2_clouds(image):
    qa = image.select('QA60')

    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    mask = (
        qa.bitwiseAnd(cloud_bit_mask).eq(0)
        .And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    )

    processed = image.updateMask(mask).divide(10000)

    # preserve metadata
    return processed.copyProperties(image, image.propertyNames())

# raw
raw_dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate(start_date, end_date)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud))
)
#processed
processed_dataset = raw_dataset.map(mask_s2_clouds)

# all available properties
sentinel_props = [
    'AOT_RETRIEVAL_ACCURACY',
    'CLOUDY_PIXEL_PERCENTAGE',
    'CLOUD_COVERAGE_ASSESSMENT',
    'CLOUDY_SHADOW_PERCENTAGE',
    'DARK_FEATURES_PERCENTAGE',
    'DATASTRIP_ID',
    'DATATAKE_IDENTIFIER',
    'DATATAKE_TYPE',
    'DEGRADED_MSI_DATA_PERCENTAGE',
    'FORMAT_CORRECTNESS',
    'GENERAL_QUALITY',
    'GENERATION_TIME',
    'GEOMETRIC_QUALITY',
    'GRANULE_ID',
    'HIGH_PROBA_CLOUDS_PERCENTAGE',
    'MEAN_SOLAR_AZIMUTH_ANGLE',
    'MEAN_SOLAR_ZENITH_ANGLE',
    'MEDIUM_PROBA_CLOUDS_PERCENTAGE',
    'MGRS_TILE',
    'NODATA_PIXEL_PERCENTAGE',
    'NOT_VEGETATED_PERCENTAGE',
    'PROCESSING_BASELINE',
    'PRODUCT_ID',
    'RADIATIVE_TRANSFER_ACCURACY',
    'RADIOMETRIC_QUALITY',
    'REFLECTANCE_CONVERSION_CORRECTION',
    'SATURATED_DEFECTIVE_PIXEL_PERCENTAGE',
    'SENSING_ORBIT_DIRECTION',
    'SENSING_ORBIT_NUMBER',
    'SENSOR_QUALITY',
    'SNOW_ICE_PERCENTAGE',
    'SPACECRAFT_NAME',
    'THIN_CIRRUS_PERCENTAGE',
    'UNCLASSIFIED_PERCENTAGE',
    'VEGETATION_PERCENTAGE',
    'WATER_PERCENTAGE',
    'WATER_VAPOUR_RETRIEVAL_ACCURACY',
    'system:index',
    'system:time_start'
]

# ---------------------------------------------------
# 8) ADD REGION-BASED BAND MEANS
# ---------------------------------------------------
def add_region_stats(image):
    stats = image.select(bands_for_stats).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=20,
        maxPixels=1e9
    )

    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')

    extra = {
        'date': date,
        'roi_lon': lon,
        'roi_lat': lat,
        'roi_buffer_m': buffer_m
    }

    # add one property per band
    for b in bands_for_stats:
        extra[f'roi_{b}_mean'] = stats.get(b)

    return image.set(extra)

raw_with_stats = raw_dataset.map(add_region_stats)

# ---------------------------------------------------
# 9) CONVERT EACH IMAGE TO ONE TABLE ROW
# ---------------------------------------------------
all_props = sentinel_props + ['date', 'roi_lon', 'roi_lat', 'roi_buffer_m'] + [
    f'roi_{b}_mean' for b in bands_for_stats
]

def image_to_feature(image):
    return ee.Feature(None, image.toDictionary(all_props))

table_fc = ee.FeatureCollection(raw_with_stats.map(image_to_feature))

# ---------------------------------------------------
# 10) EXPORT TABLE TO GOOGLE DRIVE
# ---------------------------------------------------
drive_task = ee.batch.Export.table.toDrive(
    collection=table_fc,
    description='export_s2_metadata_stats',
    folder=drive_folder,
    fileNamePrefix=drive_filename,
    fileFormat='CSV'
)
drive_task.start()

print("Google Drive export started.")

# ---------------------------------------------------
# 11) ALSO DOWNLOAD SMALLER TABLES DIRECTLY TO LOCAL CSV
# ---------------------------------------------------
try:
    df = ee.data.computeFeatures({
        'expression': table_fc,
        'fileFormat': 'PANDAS_DATAFRAME'
    })

    df.to_csv(local_csv, index=False)
    print(f"Local CSV saved: {local_csv}")
    print(df.head())
except Exception as e:
    print("Local DataFrame download failed.")
    print("This can happen if the result is too large.")
    print("Use the Google Drive export instead.")
    print("Error:", e)

# ---------------------------------------------------
# 12) OPTIONAL MAP
# ---------------------------------------------------
vis = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2']
}

m = geemap.Map()
m.centerObject(roi, 10)
m.addLayer(processed_dataset.mean(), vis, 'Sentinel-2 RGB')
m.addLayer(roi, {}, 'ROI')
m.addLayer(switzerland, {}, 'Switzerland')
m